# Livrable 3 — Image Captioning avec CNN + RNN (MS-COCO 2014)

## Objectif du notebook
Ce projet vise à générer automatiquement une légende textuelle à partir d’une image, à l’aide d’un réseau de neurones combinant :
- Un **CNN (InceptionV3)** pour l’extraction des caractéristiques visuelles (feature extraction),

InceptionV3 est pré-traitement, un réseau de neurones à convolution profond entraîné sur le jeu de données ImageNet pour la classification. L'utilisation d'un modèle pré-entraîné (Transfer Learning) permet d'exploiter des caractéristiques visuelles déjà apprises.

- Un **RNN (LSTM)** pour la génération séquentielle du texte décrivant l’image.

## Plan du notebook
1. Chargement et préparation du dataset (MS-COCO 2014)
2. Extraction des features d’images avec un CNN pré-entraîné
3. Prétraitement des légendes et tokenisation
4. Conception du modèle RNN pour la génération de texte
5. Entraînement et visualisation des performances
6. Génération d’exemples de légendes


# SETUP

In [3]:
# --- Imports Globaux ---

import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pickle
import string
import os
import json

from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Embedding, LSTM, Dropout, Add
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical, plot_model

from tqdm import tqdm

In [4]:
# --- Définition de tous les chemins ---

INPUT_DIR = "/kaggle/input/livrable3-features"
CAPTIONS_FILE = os.path.join(INPUT_DIR, 'captions.txt')
FEATURES_FILE = os.path.join(INPUT_DIR, 'features.pkl')

# Chemins de SORTIE
OUTPUT_DIR = "/kaggle/working/"
TOKENIZER_FILE = os.path.join(OUTPUT_DIR, 'tokenizer.pkl')
MODEL_FILE = os.path.join(OUTPUT_DIR, 'image_captioning_model.h5')
CURVES_FILE = os.path.join(OUTPUT_DIR, 'training_curves.png')
MODEL_PLOT_FILE = os.path.join(OUTPUT_DIR, 'model_rnn_decoder.png')

# --- 3. Vérification des Chemins ---
print("--- Vérification des chemins d'entrée ---")
if not os.path.exists(FEATURES_FILE):
    print(f"ERREUR : Le fichier 'features.pkl' est introuvable à cet emplacement : {FEATURES_FILE}")
    print("Veuillez vérifier que 'NOTEBOOK_1_OUTPUT_NAME' est correct.")
else:
    print(f"Fichier 'features.pkl' trouvé. Prêt à continuer.")

if not os.path.exists(CAPTIONS_FILE):
    print(f"ERREUR : Le fichier 'captions.txt' est introuvable à cet emplacement : {CAPTIONS_FILE}")
else:
    print(f"Fichier 'captions.txt' trouvé. Prêt à continuer.")

print(f"\nLes sorties seront sauvegardées dans : {OUTPUT_DIR}")

--- Vérification des chemins d'entrée ---
ERREUR : Le fichier 'features.pkl' est introuvable à cet emplacement : /kaggle/input/livrable3-features\features.pkl
Veuillez vérifier que 'NOTEBOOK_1_OUTPUT_NAME' est correct.
ERREUR : Le fichier 'captions.txt' est introuvable à cet emplacement : /kaggle/input/livrable3-features\captions.txt

Les sorties seront sauvegardées dans : /kaggle/working/


## KAGEL

In [5]:
KAGGLE_DATASET_NAME = "ms-coco2014"

# --- Définition de tous les chemins ---
# Chemins d'entrée (Lecture depuis le dataset Kaggle)
INPUT_DIR = f"/kaggle/input/{KAGGLE_DATASET_NAME}"
IMAGE_DIR = os.path.join(INPUT_DIR, 'train2014')
ANNOTATIONS_FILE = os.path.join(INPUT_DIR, 'annotations/captions_train2014.json')

In [6]:
# Chemins de sortie (Écriture dans le dossier de travail Kaggle)
OUTPUT_DIR = "/kaggle/working/"
CAPTIONS_FILE = os.path.join(OUTPUT_DIR, 'captions.txt')
FEATURES_FILE = os.path.join(OUTPUT_DIR, 'features.pkl')
TOKENIZER_FILE = os.path.join(OUTPUT_DIR, 'tokenizer.pkl')
MODEL_FILE = os.path.join(OUTPUT_DIR, 'image_captioning_model.h5')
CURVES_FILE = os.path.join(OUTPUT_DIR, 'training_curves.png')
MODEL_PLOT_FILE = os.path.join(OUTPUT_DIR, 'model_rnn_decoder.png')

In [7]:
print(f"Chemin des images (vérification) : {IMAGE_DIR}")
print(f"Chemin des annotations (vérification) : {ANNOTATIONS_FILE}")
print(f"Chemin de sortie (vérification) : {OUTPUT_DIR}")
# Vérifions que les chemins sont corrects
if not os.path.exists(IMAGE_DIR):
    print(f"ERREUR : Le chemin des images '{IMAGE_DIR}' est incorrect. Le dossier 'train2014' existe-t-il bien dans '{INPUT_DIR}' ?")
if not os.path.exists(ANNOTATIONS_FILE):
    print(f"ERREUR : Le chemin des annotations '{ANNOTATIONS_FILE}' est incorrect.")

Chemin des images (vérification) : /kaggle/input/ms-coco2014\train2014
Chemin des annotations (vérification) : /kaggle/input/ms-coco2014\annotations/captions_train2014.json
Chemin de sortie (vérification) : /kaggle/working/
ERREUR : Le chemin des images '/kaggle/input/ms-coco2014\train2014' est incorrect. Le dossier 'train2014' existe-t-il bien dans '/kaggle/input/ms-coco2014' ?
ERREUR : Le chemin des annotations '/kaggle/input/ms-coco2014\annotations/captions_train2014.json' est incorrect.


In [8]:
# Vérification GPU
print("--- Vérification du GPU ---")
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    print("ATTENTION : Aucun GPU n'est détecté. L'entraînement sera extrêmement lente.")
else:
    print(f"GPU détecté : {gpus[0].name}")

--- Vérification du GPU ---
ATTENTION : Aucun GPU n'est détecté. L'entraînement sera extrêmement lente.


# 1. Chargement et préparation du dataset (MS-COCO 2014)

Le réseau de neurones pour l'Image Captioning est une architecture Encodeur-Décodeur où :

L'Encodeur (CNN) : Traite l'image.

Le Décodeur (RNN) : Génère la légende séquentiellement, mot par mot.

Pour cela nous allons devoir fair un prétraitenement des données

Pré-traitement du Texte (Annotations)
Le traitement du texte (légendes) suit les étapes standard du NLP :

## Nettoyage/Tokenization :

1. Les légendes sont mises en minuscules.

- La ponctuation et les caractères spéciaux sont retirés (sauf les séparateurs de mots).
- Les légendes sont divisées en mots (tokenization).
- Des jetons de début (<start>) et de fin (<end>) sont ajoutés à chaque légende pour délimiter la séquence lors de l'entraînement.

2. Construction du Vocabulaire :

- Un dictionnaire (ou Tokenizer) est créé pour mapper chaque mot unique à un index numérique (extraction des mots les plus communs). Les mots rares peuvent être ignorés pour réduire la taille du vocabulaire (vocab_size).
- Le Tokenizer de Keras mappe chaque mot unique à un index numérique, ne conservant que les mots les plus fréquents (TOP_K) pour réduire la complexité.

3. Numérisation et Padding :

 - Chaque mot de la légende est remplacé par son index numérique (représentation vectorielle).
 - Toutes les séquences sont ramenées à une longueur maximale uniforme (max_length) en ajoutant des jetons de remplissage (padding) aux séquences plus courtes.
  - Vectorisation & Padding : Les mots sont convertis en séquences d'index (forme vectorielle). Toutes les séquences sont ensuite ajustées à la même longueur maximale (MAX_LENGTH) en ajoutant des zéros de remplissage (padding).


In [9]:
# --PARAM2TRE --
TARGET_SIZE = (299, 299)
BATCH_SIZE = 64
EMBEDDING_DIM = 256
LSTM_UNITS = 256
EPOCHS = 10
FEATURE_DIM = 2048 # Longueur du vecteur de caractéristiques InceptionV3

On vas dans un premier temps définir tout nos fonction pour Charger Nettoyer nos donnée et Features
avec la créeation de no token


In [10]:
# --- 1. Fonctions de Chargement et Préparation ---

def load_captions(filename):
    captions_dict = {}
    with open(filename, 'r') as f:
        for line in f:
            tokens = line.split()
            if len(line) < 2: continue
            image_id, image_caption = tokens[0], tokens[1:]
            image_caption = ' '.join(image_caption)
            if image_id not in captions_dict:
                captions_dict[image_id] = []
            captions_dict[image_id].append(image_caption)
    return captions_dict

def clean_captions(captions_dict):
    table = str.maketrans('', '', string.punctuation)
    for key, caption_list in captions_dict.items():
        for i in range(len(caption_list)):
            caption = caption_list[i]
            caption = caption.split()
            caption = [word.lower() for word in caption]
            caption = [w.translate(table) for w in caption]
            caption = [word for word in caption if len(word) > 1]
            caption = [word for word in caption if word.isalpha()]
            caption_list[i] = '<start> ' + ' '.join(caption) + ' <end>'
    return captions_dict

def load_image_features(filename):
    with open(filename, 'rb') as f:
        features = pickle.load(f)
    return features

def to_vocabulary(captions_dict):
    all_captions = set()
    for key in captions_dict.keys():
        [all_captions.add(d) for d in captions_dict[key]]
    return list(all_captions)

def create_tokenizer(captions_list):
    tokenizer = Tokenizer(num_words=10000, oov_token="<unk>")
    tokenizer.fit_on_texts(captions_list)
    vocab_size = len(tokenizer.word_index) + 1
    return tokenizer, vocab_size

def get_max_length(captions_dict):
    all_captions = to_vocabulary(captions_dict)
    return max(len(d.split()) for d in all_captions)


In [ ]:
print("--- 1. Chargement des données (captions.txt) ---")
captions_dict = load_captions(CAPTIONS_FILE)
print(f"Légendes chargées : {len(captions_dict)} images.")

print("--- 2. Nettoyage des légendes ---")
captions_dict = clean_captions(captions_dict)

print("--- 3. Chargement des features (features.pkl) ---")
features_dict = load_image_features(FEATURES_FILE)
print(f"Features chargées : {len(features_dict)} images.")

print("--- 4. Synchronisation données ---")
captions_train = {}
for img_id, caps in captions_dict.items():
    if img_id in features_dict:
        captions_train[img_id] = caps
print(f"Données d'entraînement finales : {len(captions_train)} images.")

print("--- 5. Création du Tokenizer ---")
all_captions_list = to_vocabulary(captions_train)
tokenizer, vocab_size = create_tokenizer(all_captions_list)
max_length = get_max_length(captions_train)
print(f"Taille du vocabulaire : {vocab_size}")
print(f"Longueur max des légendes : {max_length}")

print(f"Sauvegarde du tokenizer dans {TOKENIZER_FILE}...")
with open(TOKENIZER_FILE, 'wb') as f:
    pickle.dump(tokenizer, f)

### Générateur de Données

préparer dynamiquement les données d'entraînement du modèle de génération de légendes d’images (image captioning).
Cette fonction joue un rôle crucial dans la phase d’apprentissage, en générant à la volée les couples (image, séquence de mots) et leurs étiquettes associées.

In [14]:
def data_generator(captions_dict, features_dict, tokenizer, max_length, vocab_size, batch_size):
    X1_batch, X2_batch, y_batch = [], [], []
    n = 0
    while True:
        for image_id, caption_list in captions_dict.items():
            if image_id not in features_dict:
                continue
            image_features = features_dict[image_id]
            for caption in caption_list:
                n += 1
                sequence = tokenizer.texts_to_sequences([caption])[0]
                for i in range(1, len(sequence)):
                    X1_batch.append(image_features)
                    in_seq = sequence[:i]
                    in_seq = pad_sequences([in_seq], maxlen=max_length, padding='post')[0]
                    X2_batch.append(in_seq)
                    out_word = sequence[i]
                    out_word_one_hot = to_categorical([out_word], num_classes=vocab_size)[0]
                    y_batch.append(out_word_one_hot)
                if n == batch_size:
                    # FIX : Yield des inputs sous forme de TUPLE, pas de liste
                    yield ((np.array(X1_batch), np.array(X2_batch)), np.array(y_batch))
                    X1_batch, X2_batch, y_batch = [], [], []
                    n = 0

Ce générateur simule le processus de génération de phrase mot par mot, tout en associant chaque mot à sa représentation visuelle issue de l’image.
Il permet au modèle d’apprendre la correspondance entre le contenu visuel et les structures linguistiques.

la création du modèle d’encodage d’images à partir du réseau InceptionV3 pré-entraîné sur ImageNet.

Cette étape transforme chaque image brute en une représentation vectorielle compacte, utilisable par la partie séquentielle (RNN + Attention) du modèle pour générer des légendes.
InceptionV3 agit ici comme un encodeur visuel, permettant au réseau de capturer les éléments sémantiques essentiels de l’image.

In [ ]:
# Charger le modèle InceptionV3 pré-entraîné sur ImageNet, sans la dernière couche (top)
# Ce modèle sert d'ENCODEUR
model_inception = InceptionV3(weights='imagenet', include_top=False)
# La sortie est la couche avant la classification (par exemple 'avg_pool' pour obtenir un vecteur de 2048)
# On définit un nouveau modèle qui sort les caractéristiques
model_encoder = Model(inputs=model_inception.input, outputs=model_inception.layers[-2].output)
FEATURE_VECTOR_LENGTH = model_encoder.output_shape[1]

# Fonction de chargement et de pré-traitement d'une image
def extract_image_features(image_path):
    # 1. Charger et redimensionner l'image (299x299)
    img = load_img(image_path, target_size=TARGET_SIZE)
    img = img_to_array(img)
    # 2. Ajouter une dimension pour le lot (batch size)
    img = np.expand_dims(img, axis=0)
    # 3. Pré-traiter les pixels (normalisation spécifique à InceptionV3)
    img = preprocess_input(img)
    # 4. Extraire les caractéristiques
    feature_vector = model_encoder.predict(img, verbose=0)
    # 5. Renvoyer le vecteur de caractéristiques plat (2048)
    return feature_vector.flatten()

In [ ]:
# CODE : Fonction de Génération de Légende et Affichage des Résultats

def generate_caption(model, tokenizer, image_features, max_length):
    # Initialiser la séquence avec le jeton de départ
    in_text = '<start>'
    # Itérer jusqu'à la longueur maximale ou la détection du jeton de fin
    for i in range(max_length):
        # Convertir la séquence de mots en index numériques et ajouter le padding
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = tf.keras.preprocessing.sequence.pad_sequences([sequence], maxlen=max_length)[0]

        # Prédire le mot suivant (probabilités)
        yhat = model.predict([image_features, np.array([sequence])], verbose=0)
        # Convertir les probabilités en index (choix du mot le plus probable)
        word_index = np.argmax(yhat)
        # Mapper l'index au mot
        word = tokenizer.index_word.get(word_index, None)

        # Arrêter si le mot est None ou le jeton de fin
        if word is None or word == '<end>':
            break

        # Ajouter le mot à la séquence de sortie
        in_text += ' ' + word

    # Nettoyer et retourner la légende finale
    final_caption = in_text.replace('<start>', '').replace('<end>', '').strip()
    return final_caption


#Charger l'image test, extraire les caractéristiques
test_image_path = "path/to/test_image.jpg"
features = extract_image_features(test_image_path)
generated_caption = generate_caption(model, tokenizer, features.reshape(1, FEATURE_VECTOR_LENGTH), MAX_LENGTH)

# Afficher l'image et la légende
plt.imshow(plt.imread(test_image_path))
plt.title(f"Légende Générée: {generated_caption}")
plt.show()

# 3. Description détaillée de l’architecture du réseau de neurones

## 3.1 Vue d’ensemble

Le modèle de **captioning** repose sur une architecture **Encoder–Decoder** combinant :
- Un **CNN (InceptionV3)** servant d’**encodeur visuel** pour extraire les caractéristiques de l’image.
- Un **RNN (LSTM)** servant de **décodeur textuel**, chargé de générer séquentiellement la légende mot par mot à partir des features d’image.

Cette approche permet au modèle de comprendre le contenu visuel et de le traduire en une séquence linguistique cohérente.


In [ ]:
def build_rnn_decoder(vocab_size, max_length, embedding_dim, lstm_units, feature_dim):
    input_features = Input(shape=(feature_dim,))
    fe1 = Dropout(0.4)(input_features)
    fe2 = Dense(256, activation='relu')(fe1)

    input_sequence = Input(shape=(max_length,))
    se1 = Embedding(vocab_size, embedding_dim, mask_zero=True)(input_sequence)
    se2 = Dropout(0.4)(se1)
    se3 = LSTM(256)(se2)

    decoder1 = Add()([fe2, se3])
    decoder2 = Dense(256, activation='relu')(decoder1)
    outputs = Dense(vocab_size, activation='softmax')(decoder2)

    model = Model(inputs=[input_features, input_sequence], outputs=outputs)
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    model.summary()
    return model

model = build_rnn_decoder(vocab_size, max_length, EMBEDDING_DIM, LSTM_UNITS, FEATURE_DIM)
model.summary()
plot_model(model, to_file=MODEL_PLOT_FILE, show_shapes=True)

## 3.2 Détail de l’architecture du réseau

### **Partie CNN – Encodeur d’image**
- **Modèle utilisé** : *InceptionV3* pré-entraîné sur ImageNet.
- **Type de couches** :
  - Convolutions + Pooling internes (issues du modèle InceptionV3).
  - Couches finales supprimées (`include_top=False`) afin d’obtenir un vecteur de caractéristiques au lieu d’une prédiction de classe.
- **Sortie** : vecteur de dimension **2048**, représentant les caractéristiques visuelles globales de l’image.
- **Rôle** : Transformer chaque image en une représentation vectorielle compacte et informative.


In [ ]:
def get_cnn_model():
    base_model = InceptionV3(weights='imagenet', include_top=False, pooling='avg')
    cnn_model = Model(inputs=base_model.input, outputs=base_model.output, name="InceptionV3_Encoder")
    cnn_model.trainable = False
    return cnn_model

#### **Structure détaillée :**

| Type de couche | Détails | Rôle |
|-----------------|----------|------|
| `Input(shape=(2048,))` | Entrée des features d’image | Représentation visuelle initiale |
| `Dropout(0.4)` | Désactivation aléatoire de neurones | Réduit le surapprentissage |
| `Dense(256, activation='relu')` | Couche entièrement connectée | Apprentissage des relations non linéaires entre les features |

| Type de couche | Détails | Rôle |
|-----------------|----------|------|
| `Input(shape=(max_length,))` | Entrée séquentielle du texte | Reçoit les tokens de légende |
| `Embedding(vocab_size, embedding_dim)` | 256 dimensions d’embedding | Représentation vectorielle des mots |
| `Dropout(0.4)` | Régularisation | Empêche le surapprentissage |
| `LSTM(256)` | 256 cellules mémoire | Capture les dépendances temporelles du texte |

#### **Fusion et sortie**
| Étape | Détails | Rôle |
|--------|----------|------|
| `Add()` | Combine les sorties du CNN et du LSTM | Fusionne vision et langage |
| `Dense(256, activation='relu')` | Intégration multimodale | Apprentissage conjoint image + texte |
| `Dense(vocab_size, activation='softmax')` | Couche de sortie | Prédit le mot suivant dans la séquence |

---


## 3.3 ENTRAÎNEMENT DU MODÈLE

In [ ]:
total_sequences = 0
for cap_list in captions_train.values():
    for cap in cap_list:
        total_sequences += len(cap.split()) - 1
steps_per_epoch = total_sequences // BATCH_SIZE
if total_sequences % BATCH_SIZE != 0: steps_per_epoch += 1

In [ ]:
print(f"Total Séquences: {total_sequences}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Steps per Epoch: {steps_per_epoch}")

In [ ]:
# --- FIX : Création du tf.data.Dataset ---
# 1. Définir la signature de sortie (correspond à notre yield)
output_signature = (
    (tf.TensorSpec(shape=(None, FEATURE_DIM), dtype=tf.float32),  # X1: features
     tf.TensorSpec(shape=(None, max_length), dtype=tf.int32)),  # X2: sequence
    tf.TensorSpec(shape=(None, vocab_size), dtype=tf.float32)    # y: one-hot word
)

# 2. Créer l'objet tf.data.Dataset
train_dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(captions_train, features_dict, tokenizer, max_length, vocab_size, BATCH_SIZE),
    output_signature=output_signature
)

# 3. Optimiser les performances
train_dataset = train_dataset.prefetch(buffer_size=tf.data.AUTOTUNE)

# 4. Appeler model.fit() avec le nouveau Dataset
history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    verbose=1
)

## 3.4 Synthèse

- **Nombre total de couches principales** : 8 (hors couches internes d’InceptionV3).
- **Nombre de cellules LSTM** : 256.
- **Taille du vocabulaire** : dépend du tokenizer (≈10 000 mots dans notre cas).
- **Dimension des embeddings** : 256.
- **Fonction de perte** : *categorical_crossentropy*.
- **Optimiseur** : *Adam*.

## 3.5 MODÈLE INCEPETION V3

In [ ]:
def extract_image_features(image_path, model):
    try:
        img = load_img(image_path, target_size=(299, 299))
        img_array = img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        img_preprocessed = preprocess_input(img_array)
        features = model.predict(img_preprocessed, verbose=0)
        return features.flatten()
    except Exception as e:
        print(f"Erreur lors du traitement de {image_path}: {e}")
        return None

extract_image_features(image_path=test_image_path, model=get_cnn_model())

In [ ]:
def main_preprocessing():
    print("Chargement du modèle InceptionV3...")
    model = get_cnn_model()
    print("Modèle chargé.")

    try:
        all_images = [f for f in os.listdir(IMAGE_DIR) if f.endswith('.jpg')]
    except FileNotFoundError:
        print(f"ERREUR : Dossier images non trouvé '{IMAGE_DIR}'. Vérifiez votre variable 'KAGGLE_DATASET_NAME'.")
        return

    print(f"Total d'images trouvées : {len(all_images)}")
    if len(all_images) == 0:
        print("ERREUR : 0 images trouvées. Le chemin est probablement incorrect.")
        return

    features_dict = {}

    # La vitesse ici devrait être > 100 it/s
    for image_name in tqdm(all_images, desc="Extraction (Vitesse Kaggle)"):
        image_path = os.path.join(IMAGE_DIR, image_name)
        features = extract_image_features(image_path, model)
        if features is not None:
            features_dict[image_name] = features

    print(f"\nExtraction terminée. {len(features_dict)} features extraites.")
    print(f"Sauvegarde dans {FEATURES_FILE}...")
    with open(FEATURES_FILE, 'wb') as f:
        pickle.dump(features_dict, f)

    print(f"\n--- Étape 2 (Images) Terminée. '{FEATURES_FILE}' est prêt. ---")

# Exécuter l'étape 2
main_preprocessing()

# 4. Visualisation et Affichage des Performances

### Évolution de la Perte et de la Précision

L'évolution des performances pendant l'entraînement est cruciale pour diagnostiquer les problèmes (surapprentissage, sous-apprentissage). Les courbes de performance sont générées à partir de l'objet history retourné par la fonction model.fit().

In [ ]:
# ... (Le code d'entraînement model.fit(...) génère l'objet history)
# history = model.fit(...)

CURVES_FILE = os.path.join(OUTPUT_DIR, 'training_curves.png')

print("--- 9. Sauvegarde et Affichage des courbes d'entraînement ---")
plt.figure(figsize=(10, 4))
# Courbe de Perte (Loss)
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Perte')
plt.title('Évolution de la Perte (Loss)')
plt.xlabel('Époque')
plt.ylabel('Loss')
plt.legend()
# Courbe de Précision (Accuracy)
plt.subplot(1, 2, 2)
# La métrique 'accuracy' montre la précision à prédire le mot suivant
plt.plot(history.history['accuracy'], label='Précision')
plt.title('Évolution de la Précision (Accuracy)')
plt.xlabel('Époque')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.show() # Affiche dans le notebook
plt.savefig(CURVES_FILE) # Sauvegarde le fichier